## Exercise 1: Set Up the TelConnect Unity Catalog Structure

In this exercise, you prepare the Unity Catalog foundation that the Lakeflow Job tasks will read from and write to. You create a catalog with three schemas following a medallion architecture pattern:

- `bronze` — raw CDR records as ingested
- `silver` — cleaned and validated CDR data
- `gold` — aggregated network usage metrics for reporting

### ✅ Provided: Accept the job parameter for the processing date

When this notebook runs as a Lakeflow Job task, the job passes the `processing_date` parameter automatically. The `dbutils.widgets` API makes that value available inside the notebook.

Run the cell below to register the widget. When running manually, it defaults to today's date.

In [ ]:
# ✅ Run this cell — registers the job parameter widget

import datetime

dbutils.widgets.text("processing_date", str(datetime.date.today()), "Processing Date (YYYY-MM-DD)")
processing_date = dbutils.widgets.get("processing_date")

print(f"Processing date: {processing_date}")

### ✅ Provided: Create the Unity Catalog structure

Run the cell below to create the `telconnect_lab` catalog and the three medallion schemas. You will use these throughout the lab.


In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ Run this cell — creates the Unity Catalog structure

catalog = "telconnect_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'TelConnect network analytics platform'")
print(f"✅ Catalog '{catalog}' created.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT 'Raw ingested Call Detail Records'")
print(f"✅ Schema '{catalog}.bronze' created.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT 'Cleaned and validated CDR data'")
print(f"✅ Schema '{catalog}.silver' created.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold COMMENT 'Aggregated network usage metrics for reporting'")
print(f"✅ Schema '{catalog}.gold' created.")


## Exercise 2: Ingest Raw CDR Data (Bronze Layer)

The Bronze layer stores raw Call Detail Records exactly as received. A sample CSV representing network events for a single processing day is provided. You load it into the workspace, register it as a Bronze Delta table, and verify the record count.

### ✅ Provided: Create the raw_uploads volume and download sample data

Run the cell below to create the Unity Catalog volume and automatically download `telecom_cdrs.csv` from GitHub into the volume.


In [ ]:
# ✅ Run this cell — creates the raw_uploads volume and downloads the sample CDR data

import urllib.request

spark.sql("CREATE VOLUME IF NOT EXISTS telconnect_lab.bronze.raw_uploads COMMENT 'Landing zone for raw CDR files'")
print("✅ Volume telconnect_lab.bronze.raw_uploads is ready.")

url = "https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Allfiles/data/telecom_cdrs.csv"
dest = "/Volumes/telconnect_lab/bronze/raw_uploads/telecom_cdrs.csv"
urllib.request.urlretrieve(url, dest)
print(f"✅ telecom_cdrs.csv downloaded to {dest}")


### ✅ Provided: Load the CDR CSV into a DataFrame

Run the cell below to read the uploaded CSV into a Spark DataFrame. Inspect the output — note which records contain quality issues that you will clean in Exercise 3.


In [ ]:
# ✅ Run this cell — loads the raw CDR CSV from the volume

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

cdr_schema = StructType([
    StructField("call_id",           StringType(),  True),
    StructField("caller_msisdn",     StringType(),  True),
    StructField("callee_msisdn",     StringType(),  True),
    StructField("call_start_time",   StringType(),  True),
    StructField("call_end_time",     StringType(),  True),
    StructField("duration_seconds",  IntegerType(), True),
    StructField("call_type",         StringType(),  True),
    StructField("call_status",       StringType(),  True),
    StructField("network_type",      StringType(),  True),
    StructField("cell_tower_id",     StringType(),  True),
    StructField("region",            StringType(),  True),
    StructField("roaming",           StringType(),  True),
    StructField("data_mb",           DoubleType(),  True),
    StructField("sms_count",         IntegerType(), True),
])

raw_df = (
    spark.read
    .option("header", True)
    .schema(cdr_schema)
    .csv("/Volumes/telconnect_lab/bronze/raw_uploads/telecom_cdrs.csv")
)

raw_df.display()
print(f"\nRaw CDR records loaded: {raw_df.count()}")


### ✅ Provided: Write raw CDR data to a Bronze Delta table

Run the cell below to persist `raw_df` as a managed Delta table in Unity Catalog. This is the starting point for all downstream cleaning and aggregation.


In [ ]:
# ✅ Run this cell — writes raw CDR data to the Bronze Delta table

raw_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.bronze.cdr_bronze")

count = spark.table("telconnect_lab.bronze.cdr_bronze").count()
print(f"✅ Bronze table written. Record count: {count}")


## Exercise 3: Clean and Validate CDR Data (Silver Layer)

The Silver layer contains CDRs that have passed validation. In a production pipeline, only clean records flow downstream to aggregations and billing systems.

Review the raw data carefully. You will find the following quality issues that must be corrected:

| Issue | Example | Handling rule |
|---|---|---|
| Negative duration | CDR010 | Drop records where `duration_seconds < 0` |
| Missing callee for VOICE/DATA calls | CDR007 | Drop VOICE and DATA records where `callee_msisdn` is null |
| Zero-duration completed calls | Several | Drop VOICE records with `call_status = COMPLETED` and `duration_seconds = 0` |
| Invalid call_type value | Check data | Keep only: `VOICE`, `DATA`, `SMS` |

In [ ]:
# ✅ Run this cell — reads from the bronze table you created in Exercise 2

bronze_df = spark.table("telconnect_lab.bronze.cdr_bronze")
print(f"Bronze record count: {bronze_df.count()}")
bronze_df.display()

### ✅ Provided: Apply cleaning rules and write to Silver

Run the cell below. It applies the four data quality rules to `bronze_df` and writes the validated records to the Silver table. Review the printout to see how many records were dropped at each step.


In [ ]:
# ✅ Run this cell — applies CDR cleaning rules and writes to Silver

from pyspark.sql.functions import col

bronze_count = bronze_df.count()

silver_df = (
    bronze_df
    # Rule 1: Drop negative durations
    .filter(col("duration_seconds") >= 0)
    # Rule 2: Drop VOICE and DATA records with no callee
    .filter(
        ~((col("call_type").isin("VOICE", "DATA")) & col("callee_msisdn").isNull())
    )
    # Rule 3: Drop VOICE completed calls with zero duration
    .filter(
        ~((col("call_type") == "VOICE") & (col("call_status") == "COMPLETED") & (col("duration_seconds") == 0))
    )
    # Rule 4: Keep only valid call types
    .filter(col("call_type").isin("VOICE", "DATA", "SMS"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.silver.cdr_silver")

silver_count = spark.table("telconnect_lab.silver.cdr_silver").count()
print(f"✅ Silver table written.")
print(f"   Bronze records : {bronze_count}")
print(f"   Silver records : {silver_count}")
print(f"   Records dropped: {bronze_count - silver_count}")


## Exercise 4: Aggregate Network Metrics (Gold Layer)

The Gold layer contains aggregated metrics consumed by dashboards and operations teams. TelConnect's network operations centre needs a daily summary per region and network type showing:

- Total calls handled
- Total successful calls (`COMPLETED` status)
- Call drop rate (percentage of calls that were `DROPPED`)
- Average call duration (for `VOICE` calls only)
- Total data transferred in MB (for `DATA` calls only)

In [ ]:
# ✅ Run this cell — reads from the silver table

silver_df = spark.table("telconnect_lab.silver.cdr_silver")
print(f"Silver record count: {silver_df.count()}")

### ✅ Provided: Build the regional network summary

Run the cell below. It aggregates the Silver CDR data by region and network type, computing the five operational metrics, and writes the result to the Gold table.


In [ ]:
# ✅ Run this cell — aggregates CDR Silver data into the Gold network summary

from pyspark.sql.functions import (
    count, sum as spark_sum, avg, when, round as spark_round
)

gold_df = (
    silver_df
    .groupBy("region", "network_type")
    .agg(
        count("*").alias("total_calls"),
        count(when(col("call_status") == "COMPLETED", 1)).alias("completed_calls"),
        spark_round(
            count(when(col("call_status") == "DROPPED", 1)) * 100.0 / count("*"), 2
        ).alias("drop_rate_pct"),
        spark_round(
            avg(when((col("call_type") == "VOICE") & (col("duration_seconds") > 0), col("duration_seconds"))), 1
        ).alias("avg_voice_duration_sec"),
        spark_round(
            spark_sum(when(col("call_type") == "DATA", col("data_mb"))), 2
        ).alias("total_data_mb"),
    )
    .orderBy("region", "network_type")
)

gold_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.gold.network_summary")

print(f"✅ Gold table written. Rows: {spark.table('telconnect_lab.gold.network_summary').count()}")
spark.table("telconnect_lab.gold.network_summary").display()


## Exercise 5: Make the Notebook Job-Ready

When this notebook runs as a Lakeflow Job task, it should communicate outcomes back to the job. Databricks provides two mechanisms:

- **`dbutils.notebook.exit()`** — exits the notebook and passes a string value to any task that calls this notebook via the `Run Notebook` task type.
- **`dbutils.jobs.taskValues.set()`** — stores a named value that downstream tasks in the same job run can retrieve.

You use both here to make the notebook observable and chainable.

### ✅ Provided: Count Gold records and share the result with the job

Run the cell below. It publishes the Gold row count as a task value (so downstream job tasks can read it) and exits the notebook with a structured JSON status string. This is what makes the notebook behave as a proper Lakeflow Job task.


In [ ]:
# ✅ Run this cell — publishes results and exits the notebook as a job task

import json

gold_row_count = spark.table("telconnect_lab.gold.network_summary").count()

# Share the count with downstream tasks in the same job run
dbutils.jobs.taskValues.set(key="gold_row_count", value=gold_row_count)

print(f"✅ Pipeline complete for processing_date={processing_date}. Gold rows: {gold_row_count}")

# Exit with a structured status — readable from the job run timeline
result = json.dumps({"status": "success", "processing_date": processing_date, "gold_rows": gold_row_count})
dbutils.notebook.exit(result)
